# 09 · Voice

The voice chat from Plan Section 6: a thin layer around the same graph and `ChatSession` the
text chat uses, so nothing in the agents changes. The conversation is `ChatSession`'s: the
greeting, small talk, the request, a clarifying question if needed, the "just to confirm"
read-back, then the search (notebook 07).

| Step | How | If it fails |
|---|---|---|
| Greet | "Hi! I'm your AI card-shopping assistant. How are you doing today?", spoken at startup | |
| Record | Enter to start, Enter to stop; `sounddevice`, 16 kHz mono | type instead (`--keyboard`) |
| Transcribe | Groq `whisper-large-v3-turbo`, with the card and set names as a vocabulary prompt | Gemini transcribes the same audio, then you type it |
| Run the agents | `ChatSession.send(text, on_task=...)` streams the graph node by node | |
| Live search lines | with `--live`: a casual line when the search starts ("Hold up a sec while I swim the internet for you!"), one as each step finishes, and a "still swimming" line after 25 s of quiet | |
| Filler | one of 5 short clips, made once at startup, starts when EVALUATING starts | |
| Speak | Edge-TTS (`en-US-AvaNeural`), sentence by sentence: each one is made while the one before plays | after an error or 3 s with no audio: macOS `say` (pyttsx3 elsewhere), then text |
| Timing log | one row per turn in `eval/voice_runs.jsonl` | |

The timing log measures from the moment you stop talking: to the transcript, to the first
filler audio and to the first answer audio, plus each graph stage. Filler and answer times are
kept apart, because the filler shortens the silence, not the wait for the answer (H5). The
voice chat doesn't use the LLM cache, so every time in the log is a real call.

Run it with `python -m src --voice`, or Run and Debug → "Voice chat (Gemini)". On a Mac, VS Code
needs microphone permission (System Settings → Privacy & Security → Microphone); without it the
recording is silent, and the chat tells you so.

The plan's local fallback for speech to text (faster-whisper) can't be installed on Python 3.14
on a Mac (its engine, ctranslate2, has no 3.14 build there), so Gemini is the fallback.

In [ ]:
#| default_exp voice

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

In [ ]:
#| export
"""Voice chat (Plan Section 6): push-to-talk recording, Whisper speech-to-text, the same graph
as the text chat, a filler line while the Merchant checks the listings, and Edge-TTS for the
reply. A thin layer around `ChatSession`: the agents don't know whether the buyer typed or spoke.

    python -m src --voice              # Enter to talk, Enter to stop
    python -m src --voice --keyboard   # type the request, hear the reply
    python -m src --voice --audio q.wav --mute   # a recording as the request, reply printed

Every turn appends one row to `voice.timing_log` (eval/voice_runs.jsonl): seconds from the end
of the recording to the transcript, to the first filler audio and to the first answer audio,
plus each graph stage. Time to filler and time to answer are kept apart (hypothesis H5).
"""

import argparse
import asyncio
import hashlib
import importlib.util
import io
import json
import re
import shutil
import subprocess
import tempfile
import threading
import time
import wave
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np

from src.config import env, load_config, repo_path
from src.graph import ChatSession, add_live_options, build_graph, live_mode
from src.llm import LLMError, make_llm
from src.memory import BuyerMemory
from src.pipeline import load_data
from src.ui import CardDetails, LiveBoard, open_page, run_info

In [ ]:
#| export
MIC_HINT = (
    "The recording was silent. On a Mac, VS Code (or your terminal) needs microphone "
    "permission: System Settings → Privacy & Security → Microphone. Then restart it."
)

## Recording

Push-to-talk with `sounddevice`. The loudest sample is kept, so a silent recording (no microphone permission) is caught before it is sent anywhere.

In [ ]:
#| export
class TranscriptionError(RuntimeError):
    """Neither Groq Whisper nor the Gemini fallback produced a transcript."""

In [ ]:
#| export
@dataclass
class Recording:
    """One request as audio. `ended_at` (time.perf_counter()) is when the buyer stopped talking:
    every time in the timing log is measured from it."""

    audio: bytes  # WAV from the microphone, or a file's own bytes (wav, mp3, m4a, ...)
    filename: str  # its extension tells the speech-to-text service the format
    ended_at: float
    seconds: float | None = None  # None for files: no length or silence check
    peak: float | None = None  # loudest sample, 0-1; 0 means the microphone sent silence

    @classmethod
    def from_file(cls, path: str | Path) -> Recording:
        p = Path(path)
        return cls(p.read_bytes(), p.name, time.perf_counter())

In [ ]:
#| export
def wav_bytes(samples: np.ndarray, rate: int) -> bytes:
    """16-bit mono WAV from int16 samples (what the microphone gives and Whisper accepts)."""
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes(np.asarray(samples, dtype="<i2").tobytes())
    return buf.getvalue()

In [ ]:
#| export
class PushToTalk:
    """Enter to start, Enter to stop: 16 kHz mono through sounddevice. Simpler and more
    reliable than detecting speech automatically (Section 6)."""

    def __init__(self, config: dict[str, Any], ask: Callable[[str], str] = input):
        self.rate = int(config["voice"]["sample_rate"])
        self.ask = ask

    def record(self) -> Recording | None:
        """None when the buyer types q (quit)."""
        import sounddevice as sd

        if self.ask("\n[Enter] to talk, q + Enter to quit: ").strip().lower() in {"q", "quit"}:
            return None
        frames: list[np.ndarray] = []
        with sd.InputStream(
            samplerate=self.rate,
            channels=1,
            dtype="int16",
            callback=lambda data, _n, _t, _status: frames.append(data.copy()),
        ):
            self.ask("listening... [Enter] to stop ")
        ended = time.perf_counter()
        samples = np.concatenate(frames)[:, 0] if frames else np.zeros(0, dtype="int16")
        peak = float(np.abs(samples.astype(np.int32)).max()) / 32768 if samples.size else 0.0
        return Recording(
            wav_bytes(samples, self.rate), "speech.wav", ended, samples.size / self.rate, peak
        )

## Speech to text

Groq Whisper first; if it errors or returns nothing, Gemini transcribes the same audio.

In [ ]:
#| export
STT_PROMPT = (
    "Transcribe this recording of a buyer asking about Pokémon cards. Return only the words "
    "spoken, with prices written as $ amounts, and nothing else."
)
AUDIO_TYPES = {
    ".wav": "audio/wav",
    ".mp3": "audio/mp3",
    ".m4a": "audio/aac",
    ".aac": "audio/aac",
    ".ogg": "audio/ogg",
    ".flac": "audio/flac",
    ".aiff": "audio/aiff",
}

In [ ]:
#| export
def vocabulary_prompt(names: list[str], max_chars: int) -> str:
    """ "Pokémon cards: Charizard, Base Set, ..." with each name once, cut to `max_chars`
    (Whisper reads at most 224 tokens of prompt)."""
    seen: dict[str, None] = {}
    for name in names:
        if name and name.lower() not in {n.lower() for n in seen}:
            seen[name] = None
    text = "Pokémon cards: " + ", ".join(seen)
    if len(text) <= max_chars:
        return text
    cut = text.rfind(", ", 0, max_chars)
    return text[: cut if cut > 0 else max_chars] + "."

In [ ]:
#| export
class Transcriber:
    """Groq `models.stt` (whisper-large-v3-turbo). If Groq fails, Gemini `models.stt_fallback`
    transcribes the same audio. The plan's local faster-whisper fallback has no Python 3.14
    build for macOS, so it is not used. Clients can be passed in (the tests do).

    Both get a vocabulary prompt (`voice.stt_vocabulary` plus `names`, e.g. the card and set
    names in the listing pool): without it Whisper hears "Pikachu" as "Pocaccia", "Mewtwo" as
    "MuteWO" and "holo" as "hollow"."""

    def __init__(
        self,
        config: dict[str, Any],
        groq: Any = None,
        gemini: Any = None,
        names: list[str] | tuple[str, ...] = (),
    ):
        v = config["voice"]
        self.models = config["models"]
        self.language = v["stt_language"]
        self.vocabulary = vocabulary_prompt(
            [*v["stt_vocabulary"], *names], int(v["stt_prompt_chars"])
        )
        self._groq, self._gemini = groq, gemini

    def transcribe(self, rec: Recording) -> tuple[str, str]:
        """(transcript, "groq" or "gemini")."""
        errors = []
        for name, fn in (("groq", self._with_groq), ("gemini", self._with_gemini)):
            try:
                text = (fn(rec) or "").strip()
            except Exception as e:  # noqa: BLE001 - any failure moves on to the next service
                errors.append(f"{name}: {e}")
                continue
            if text:
                return text, name
            errors.append(f"{name}: empty transcript")
        raise TranscriptionError("; ".join(errors))

    def _with_groq(self, rec: Recording) -> str:
        if self._groq is None:
            from groq import Groq

            key = env("GROQ_API_KEY")
            if not key:
                raise TranscriptionError("GROQ_API_KEY is missing")
            self._groq = Groq(api_key=key)
        result = self._groq.audio.transcriptions.create(
            file=(rec.filename, rec.audio),
            model=self.models["stt"],
            language=self.language,
            prompt=self.vocabulary,
            temperature=0.0,
        )
        return result.text

    def _with_gemini(self, rec: Recording) -> str:
        from google.genai import types

        if self._gemini is None:
            from google import genai

            key = env("GEMINI_API_KEY")
            if not key:
                raise TranscriptionError("GEMINI_API_KEY is missing")
            self._gemini = genai.Client(api_key=key)
        mime = AUDIO_TYPES.get(Path(rec.filename).suffix.lower(), "audio/wav")
        result = self._gemini.models.generate_content(
            model=self.models["stt_fallback"],
            contents=[
                types.Part.from_bytes(data=rec.audio, mime_type=mime),
                f"{STT_PROMPT} {self.vocabulary}",
            ],
            config=types.GenerateContentConfig(temperature=0.0),
        )
        return result.text

## Text to speech and playback

Edge-TTS returns MP3, which `soundfile` decodes so `sounddevice` can play it without blocking. Any Edge-TTS failure, including the 403s it gives under load and anything slower than `limits.tts_timeout_seconds`, falls back to the local voice.

In [ ]:
#| export
@dataclass
class Clip:
    """Something to say. `source` is how it will sound: "edge-tts" (decoded samples), "say"
    (macOS voice), "pyttsx3" (local voice elsewhere) or "text" (no voice: printed only)."""

    text: str
    source: str
    samples: np.ndarray | None = None  # float32 mono
    rate: int = 24000

In [ ]:
#| export
def decode_audio(data: bytes) -> tuple[np.ndarray, int]:
    """MP3 (Edge-TTS) or WAV bytes -> float32 mono samples and sample rate."""
    import soundfile as sf

    samples, rate = sf.read(io.BytesIO(data), dtype="float32")
    return (samples.mean(axis=1) if samples.ndim == 2 else samples), rate

In [ ]:
#| export
def run_async(make_coro: Callable[[], Any]) -> Any:
    """asyncio.run, also from a notebook (whose event loop is already running)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(make_coro())
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(lambda: asyncio.run(make_coro())).result()

In [ ]:
#| export
def local_voice() -> str:
    """The fallback voice on this machine: macOS `say`, pyttsx3 if installed, or none."""
    if shutil.which("say"):
        return "say"
    return "pyttsx3" if importlib.util.find_spec("pyttsx3") else "text"

In [ ]:
#| export
class Voice:
    """Text to speech: Edge-TTS (`voice.tts_voice`), else the local voice, so the demo never
    stops mid-sentence. Edge-TTS fails over when it errors (it returns 403s under load) or sends
    nothing for `limits.tts_timeout_seconds`: a long reply can take more than 3 seconds to
    arrive in full while audio keeps coming. `clip(text, cache=True)` keeps the MP3 in
    `voice.clip_cache` (the fillers). `synth` replaces Edge-TTS (the tests pass a fake)."""

    def __init__(self, config: dict[str, Any], synth: Callable[[str], bytes] | None = None):
        v = config["voice"]
        self.name, self.rate = v["tts_voice"], v["tts_rate"]
        self.timeout = float(config["limits"]["tts_timeout_seconds"])
        self.cache_dir = repo_path(v["clip_cache"])
        self.synth = synth or self.edge_tts
        self.fallback = local_voice()
        self.last_error: str | None = None

    def edge_tts(self, text: str) -> bytes:
        import edge_tts

        async def run() -> bytes:
            audio = bytearray()
            chunks = edge_tts.Communicate(text, self.name, rate=self.rate).stream()
            while True:  # at most `timeout` seconds of waiting for each piece
                try:
                    chunk = await asyncio.wait_for(anext(chunks), self.timeout)
                except StopAsyncIteration:
                    return bytes(audio)
                if chunk["type"] == "audio":
                    audio += chunk["data"]

        return run_async(run)

    def mp3(self, text: str, cache: bool = False) -> bytes:
        """Edge-TTS audio for one line (MP3); with `cache`, kept in `voice.clip_cache`. Raises
        on any failure. The web page's voice uses this directly (notebook 10)."""
        key = hashlib.sha256(f"{self.name}|{self.rate}|{text}".encode()).hexdigest()[:16]
        path = self.cache_dir / f"{key}.mp3"
        if cache and path.exists():
            return path.read_bytes()
        audio = self.synth(text)
        if not audio:
            raise RuntimeError("Edge-TTS returned no audio")
        if cache:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(audio)
        return audio

    def clip(self, text: str, cache: bool = False) -> Clip:
        try:
            samples, rate = decode_audio(self.mp3(text, cache))
            return Clip(text, "edge-tts", samples, rate)
        except Exception as e:  # noqa: BLE001 - error, 403 or timeout: the local voice instead
            self.last_error = f"{type(e).__name__}: {e}" if str(e) else type(e).__name__
            return Clip(text, self.fallback)

In [ ]:
#| export
class Speaker:
    """Plays one clip at a time without blocking the caller; `play()` first waits for the clip
    before it, so the answer never talks over the filler. Returns when the audio started.
    `busy` says whether something is playing, so an optional line can be skipped instead of
    queued. Safe to call from the "still working" thread. If the audio device fails, it says
    why once and goes quiet instead of stopping the chat."""

    def __init__(self, mute: bool = False):
        self.mute = mute
        self.error: str | None = None
        self.last_started = 0.0
        self._playing = False
        self._proc: subprocess.Popen | None = None
        self._lock = threading.Lock()

    def play(self, clip: Clip) -> float:
        with self._lock:
            self.wait()
            started = self.last_started = time.perf_counter()
            if self.mute or clip.source == "text":
                return started
            try:
                self._start(clip)
            except Exception as e:  # noqa: BLE001 - no device, busy, ...: keep chatting
                self.error, self.mute = f"{type(e).__name__}: {e}", True
                print(f"(can't play audio: {self.error}; replies are printed only)")
            return started

    @property
    def busy(self) -> bool:
        if self._proc is not None and self._proc.poll() is None:
            return True
        if self._playing:
            import sounddevice as sd

            try:
                return bool(sd.get_stream().active)
            except Exception:  # noqa: BLE001 - no stream: nothing is playing
                return False
        return False

    def _start(self, clip: Clip) -> None:
        if clip.samples is not None:
            import sounddevice as sd

            sd.play(clip.samples, clip.rate)
            self._playing = True
        elif clip.source == "say":
            self._proc = subprocess.Popen(["say", clip.text])
        elif clip.source == "pyttsx3":
            import pyttsx3

            engine = pyttsx3.init()
            engine.say(clip.text)
            engine.runAndWait()

    def wait(self) -> None:
        if self._playing:
            import sounddevice as sd

            sd.wait()
            self._playing = False
        if self._proc is not None:
            self._proc.wait()
            self._proc = None

## One spoken turn

The lines during a live search come from the `on_task` callback (the `searching`, `scraped` and `identified` progress events) and from a small thread that fills silences; the filler starts the moment EVALUATING starts, and the answer waits for whatever is playing to finish. The reply is spoken a sentence at a time: Edge-TTS needs 2-7 seconds for a whole Gemini reply but under a second for its first sentence, and each next sentence is made while the one before it plays.

In [ ]:
#| export
def speakable(text: str) -> str:
    """What the voice reads: "$360.00" as "$360" and "4/102" as "4 of 102". Only the audio
    changes; the printed reply and the number check use the original text."""
    text = re.sub(r"(\$\d[\d,]*)\.00\b", r"\1", text)
    return re.sub(r"\b(\d{1,3})/(\d{2,3})\b", r"\1 of \2", text)

In [ ]:
#| export
def sentences(text: str) -> list[str]:
    """The reply in sentences, so the first one can play while the rest are synthesized: Edge-TTS
    takes 2-7 seconds for a whole 250-380 character reply, under a second for one sentence."""
    parts = [p for p in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9$])", text.strip()) if p]
    return parts or [text]

In [ ]:
#| export
class VoiceChat:
    """One spoken conversation around a ChatSession. All the fixed lines (fillers, the live
    search lines) are made once, here, so playing one adds no speech-synthesis delay.

    During a turn: when a live search starts, a casual search line plays ("Hold up a sec while
    I swim the internet for you!"), then a line as each step finishes, and a "still working"
    line whenever nothing has been said for `voice.still_after_s`. When EVALUATING starts, a
    filler plays (once per turn, skipped if something is already playing). The reply is spoken
    sentence by sentence, each synthesized while the one before it plays.

    With a `board` (the web page, `--ui`), everything said and every step also goes to the page:
    the transcript, the lines spoken during the search, the reply and the results."""

    def __init__(
        self,
        chat: ChatSession,
        config: dict[str, Any],
        transcriber: Transcriber,
        voice: Voice,
        speaker: Speaker,
        log_path: str | Path | None = None,
        model: str = "",
        echo: Callable[[str], None] = print,
        board: LiveBoard | None = None,
    ):
        v = config["voice"]
        self.board = board
        self.chat, self.transcriber, self.voice, self.speaker = chat, transcriber, voice, speaker
        self.min_seconds, self.silence = float(v["min_seconds"]), float(v["silence_peak"])
        self.log_path = repo_path(log_path or v["timing_log"])
        self.model, self.echo = model, echo
        self.still_after = float(v["still_after_s"])
        self.fillers = [voice.clip(line, cache=True) for line in v["fillers"]]
        self.search_lines = [voice.clip(line, cache=True) for line in v["search_lines"]]
        self.progress_lines = {k: voice.clip(t, cache=True) for k, t in v["progress_lines"].items()}
        self.still_lines = [voice.clip(line, cache=True) for line in v["still_lines"]]
        self._next = {"filler": 0, "search": 0, "still": 0}

    def _rotate(self, kind: str, clips: list[Clip]) -> Clip:
        clip = clips[self._next[kind] % len(clips)]
        self._next[kind] += 1
        return clip

    def say(self, text: str) -> float | None:
        """Speak a reply sentence by sentence; returns when its audio started."""
        parts = sentences(speakable(text))
        started = None
        for i, part in enumerate(parts):  # each made while the sentence before it plays
            at = self.speaker.play(self.voice.clip(part))
            started = at if i == 0 else started
        self.speaker.wait()
        return started

    def start(self) -> str | None:
        """Say the greeting (if the chat has one) before the buyer speaks."""
        line = self.chat.opening()
        if line:
            self.echo(f"assistant: {line}")
            if self.board:
                self.board.say("assistant", line, "voice")
            self.say(line)
        return line

    def turn(self, rec: Recording | None = None, typed: str | None = None) -> dict | None:
        """One buyer message, spoken (`rec`) or typed. Returns the timing row, or None when the
        recording was too short or silent."""
        if rec is not None and rec.seconds is not None and rec.seconds < self.min_seconds:
            self.echo("(too short: hold Enter-to-stop until you've finished speaking)")
            return None
        if rec is not None and rec.peak is not None and rec.peak < self.silence:
            self.echo(MIC_HINT)
            return None
        t0 = rec.ended_at if rec is not None else time.perf_counter()
        if rec is not None:
            text, stt = self.transcriber.transcribe(rec)
        else:
            text, stt = typed or "", "keyboard"
        t_text = time.perf_counter()
        if rec is not None:
            self.echo(f"you: {text}")  # what Whisper heard
        if self.board:
            self.board.begin_turn(text, "voice" if rec is not None else "typed")

        stages: dict[str, float] = {}
        started: dict[str, float] = {}
        heard: dict[str, Any] = {}  # first time each kind of line started, and the filler said
        searching = threading.Event()  # a live search is running: the still-working lines
        done = threading.Event()

        def line(kind: str, clip: Clip, skip_if_busy: bool = True) -> None:
            if skip_if_busy and self.speaker.busy:
                return
            at = self.speaker.play(clip)
            heard.setdefault(kind, at)
            if kind == "filler":
                heard["filler_text"] = clip.text
            if self.board:
                self.board.note(clip.text, kind)

        def still_working() -> None:
            while not done.wait(1.0):
                quiet = time.perf_counter() - self.speaker.last_started
                if searching.is_set() and quiet >= self.still_after and not self.speaker.busy:
                    line("still", self._rotate("still", self.still_lines))

        def on_task(name: str, phase: str, info: dict | None = None) -> None:
            if self.board:
                self.board.on_task(name, phase, info)
            now = time.perf_counter()
            if phase == "progress":
                if name == "searching":
                    line("search", self._rotate("search", self.search_lines), skip_if_busy=False)
                    searching.set()
                elif name in self.progress_lines:
                    line("progress", self.progress_lines[name])
                return
            if phase == "end":
                stages[name] = round(stages.get(name, 0.0) + now - started.pop(name, now), 3)
                if name == "SEARCHING":
                    searching.clear()
                return
            started[name] = now
            if name == "EVALUATING" and "filler" not in heard:
                line("filler", self._rotate("filler", self.fillers))

        ticker = threading.Thread(target=still_working, daemon=True)
        ticker.start()
        try:
            reply, state = self.chat.send(text, on_task=on_task)
        except Exception as e:
            if self.board:
                self.board.error(f"{type(e).__name__}: {e}")
            raise
        finally:
            done.set()
            ticker.join(timeout=2)
        t_reply = time.perf_counter()
        self.echo(f"assistant: {reply}")
        if self.board:  # on its own thread: card details for the page may take a moment
            threading.Thread(target=self.board.end_turn, args=(reply, state, "voice"),
                             daemon=True).start()  # fmt: skip
        self.voice.last_error = None
        parts = sentences(speakable(reply))
        clip = self.voice.clip(parts[0])
        t_tts = time.perf_counter()
        answer_at = self.speaker.play(clip)  # waits for a filler or search line to finish first
        sources = {clip.source}
        for part in parts[1:]:  # each made while the sentence before it plays
            clip = self.voice.clip(part)
            sources.add(clip.source)
            self.speaker.play(clip)
        self.speaker.wait()

        def secs(t: float | None) -> float | None:
            return None if t is None else round(t - t0, 3)

        live = state.get("live_search") or {}
        row = {
            "time": datetime.now().isoformat(timespec="seconds"),
            "session_id": self.chat.session_id,
            "model": self.model,
            "stt": stt,
            "tts": "+".join(sorted(sources)),
            "transcript": text,
            "state": state.get("current_state"),
            "reply": reply,
            "filler": heard.get("filler_text"),
            "record_s": None if rec is None or rec.seconds is None else round(rec.seconds, 2),
            "to_transcript_s": secs(t_text),
            "to_search_line_s": secs(heard.get("search")),
            "to_filler_s": secs(heard.get("filler")),
            "to_answer_s": secs(answer_at),
            "graph_s": round(t_reply - t_text, 3),
            "tts_s": round(t_tts - t_reply, 3),  # the first sentence
            "stages": stages,
            "live": {k: live[k] for k in ("scraped", "usable", "kept", "seconds") if k in live},
            "tts_error": self.voice.last_error,
        }
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        return row

## Terminal and VS Code entry point

`python -m src --voice`: `--keyboard` to type, `--mute` to print instead of speaking, `--offline` for rule-based agents (speech still uses Groq and Edge-TTS), `--live` to search Marketplace for each confirmed request, `--replay FILE` to do that with a saved scrape (free), `--audio FILE ...` to use recordings as the buyer's turns. `synthesize_requests()` makes such recordings with Edge-TTS.

In [ ]:
#| export
def main(argv: list[str] | None = None) -> int:
    """Voice chat: `python -m src --voice`. In VS Code: Run and Debug → "Voice chat (Gemini)"."""
    ap = argparse.ArgumentParser(prog="python -m src --voice", description="Push-to-talk chat")
    ap.add_argument("--voice", action="store_true", help=argparse.SUPPRESS)
    ap.add_argument(
        "--offline",
        action="store_true",
        help="rule-based agents (no Gemini); speech still uses Groq and Edge-TTS",
    )
    ap.add_argument("--keyboard", action="store_true", help="type instead of talking")
    ap.add_argument("--mute", action="store_true", help="print replies instead of speaking them")
    ap.add_argument(
        "--audio",
        nargs="+",
        metavar="FILE",
        help="use these recordings as the requests, one turn each, then exit",
    )
    ap.add_argument("--session", default="voice", help="buyer-memory session id")
    ap.add_argument("--db", help="buyer-memory SQLite file (default data/buyer_memory.sqlite)")
    ap.add_argument("--log", help="timing log (default voice.timing_log in config.yaml)")
    ap.add_argument(
        "--ui", action="store_true", help="follow the chat and the results in the browser"
    )
    ap.add_argument("--port", type=int, help="web page port with --ui (default ui.port)")
    ap.add_argument("--no-browser", action="store_true", help="with --ui: don't open a tab")
    add_live_options(ap)
    args = ap.parse_args(argv)

    cfg = load_config()
    live_mode(cfg, args.live, args.replay)
    cfg["llm"]["cache_dir"] = None  # no cached replies: every timing in the log is a real call
    llm = make_llm(cfg, "offline" if args.offline else None)
    data = load_data(cfg)
    memory = BuyerMemory(args.db) if args.db else BuyerMemory()
    app = build_graph(cfg, memory=memory, llm=llm, data=data)
    voice, speaker = Voice(cfg), Speaker(mute=args.mute)
    page = None
    if args.ui:
        info = run_info(cfg, data, llm, "voice")
        board = LiveBoard(cfg, data, CardDetails(cfg, offline=args.offline), info=info)
        page = open_page(cfg, data, llm, "voice", None, args.port,
                         False if args.no_browser else None, board)  # fmt: skip
    names = {x.card_name for x in data.listings} | {x.set_name for x in data.listings if x.set_name}
    vc = VoiceChat(
        ChatSession(app, args.session, cfg, llm=llm),
        cfg,
        Transcriber(cfg, names=sorted(names)),
        voice,
        speaker,
        log_path=args.log,
        model=llm.model,
        board=page.board if page else None,
    )
    fillers = {c.source for c in vc.fillers}
    if cfg["merchant"]["live"]:
        replay = cfg["merchant"].get("live_replay")
        where = f"live, replaying {Path(replay).name}" if replay else (
            f"live, {cfg['merchant']['live_limit']} per search")  # fmt: skip
    else:
        where = f"{len(data.listings)} ({data.source})"
    print(
        f"model: {llm.model} | listings: {where} | speech: "
        f"{cfg['models']['stt']} | voice: {voice.name} ({', '.join(sorted(fillers))})"
    )
    if voice.last_error:
        print(f"Edge-TTS failed ({voice.last_error}); using the {voice.fallback} voice.")

    def run(rec: Recording | None = None, typed: str | None = None) -> None:
        try:
            row = vc.turn(rec, typed)
        except TranscriptionError as e:
            print(f"(couldn't transcribe: {e})")
            typed = input("type it instead: ").strip()
            row = vc.turn(typed=typed) if typed else None
        except LLMError as e:
            print(f"(LLM error: {e})")
            return
        if row:
            live = row["live"]
            found = f", live search {live['seconds']}s: {live['kept']} cards" if live else ""
            print(
                f"      [to transcript {row['to_transcript_s']}s, filler {row['to_filler_s']}s, "
                f"answer {row['to_answer_s']}s{found}]"
            )

    if page:
        print(f"web page: {page.url}")
    if not args.audio:
        print("Type quit to stop." if args.keyboard else "Press Enter to talk; q to quit.")
    try:
        return talk(args, cfg, vc, run)
    finally:
        if page:
            page.close()

In [ ]:
#| export
def talk(args: argparse.Namespace, cfg: dict[str, Any], vc: VoiceChat, run) -> int:
    """The conversation loop of main(): the greeting, then a turn per recording or line."""
    vc.start()  # the greeting
    if args.audio:
        for path in args.audio:
            run(Recording.from_file(path))
        return 0
    recorder = PushToTalk(cfg)
    while True:
        try:
            if args.keyboard:
                text = input("\nyou: ").strip()
                if not text or text.lower() in {"quit", "exit"}:
                    break
                run(typed=text)
            else:
                rec = recorder.record()
                if rec is None:
                    break
                run(rec)
        except EOFError, KeyboardInterrupt:
            break
    return 0

In [ ]:
#| export
def synthesize_requests(config: dict[str, Any], texts: list[str], folder: Path | None = None):
    """Edge-TTS recordings of typed requests, for testing the voice loop without a microphone
    (and the same audio every run). Returns the MP3 paths."""
    voice = Voice(config)
    out = Path(folder or tempfile.mkdtemp())
    out.mkdir(parents=True, exist_ok=True)
    paths = []
    for i, text in enumerate(texts, 1):
        path = out / f"request_{i:02d}.mp3"
        path.write_bytes(voice.edge_tts(text))
        paths.append(path)
    return paths

# Try it

Offline by default: the "recording" is the request's text and the replies are printed, so this
runs with no keys, no network and no audio device. Set `SPEECH = True` to use Groq Whisper,
Edge-TTS and your speakers (keys in `.env`, network), and `LIVE = True` in the setup cell for
Gemini agents.

What the voice reads, compared with the printed reply:

In [ ]:
for reply in [
    "Your best option is the Charizard 4/102 for $360.00 total.",
    "It's $40.50 over budget.",
]:
    print(f"{reply!r:62} -> {speakable(reply)!r}")

A voice chat around the graph. The filler clips are made here, once:

In [ ]:
import tempfile

import pandas as pd

from src.config import load_config
from src.graph import ChatSession, build_graph
from src.llm import make_llm
from src.memory import BuyerMemory

SPEECH = False  # True: real Groq Whisper, Edge-TTS and your speakers

cfg = load_config()
llm = make_llm(cfg)
app = build_graph(cfg, llm=llm, memory=BuyerMemory(Path(tempfile.mkdtemp()) / "voice.sqlite"))
log = Path(tempfile.mkdtemp()) / "voice_runs.jsonl"


class TextAsAudio:  # offline stand-in for Whisper: the "audio" is the request's text
    def transcribe(self, rec):
        return rec.audio.decode(), "offline"


class PrintedVoice(Voice):  # offline stand-in for Edge-TTS: nothing is synthesized
    def clip(self, text, cache=False):
        return Clip(text, "text")


if SPEECH:
    transcriber, voice, speaker = Transcriber(cfg), Voice(cfg), Speaker()
else:
    transcriber, voice, speaker = TextAsAudio(), PrintedVoice(cfg), Speaker(mute=True)
vc = VoiceChat(
    ChatSession(app, "notebook", cfg, llm=llm),
    cfg,
    transcriber,
    voice,
    speaker,
    log_path=log,
    model=llm.model,
)
print("fillers:", [(c.text, c.source) for c in vc.fillers])

The whole conversation: the greeting, small talk, a spoken request (with `SPEECH = True` it is first turned into audio with Edge-TTS, so Whisper hears a real recording), the read-back and the yes. Only the last turn searches, so only it gets a filler:

In [ ]:
vc.start()
vc.turn(typed="I'm good, thanks!")
request = "A Base Set Charizard under $400, I care about condition"
if SPEECH:
    rec = Recording.from_file(synthesize_requests(cfg, [request])[0])
else:
    rec = Recording(request.encode(), "request.txt", time.perf_counter())
vc.turn(rec)
row = vc.turn(typed="yes")
pd.Series(row)

A clarifying question gets no filler either (EVALUATING doesn't run); the read-back and the search follow:

In [ ]:
for text in ["I want a Charizard", "Base Set, under $400", "yep"]:
    row = vc.turn(typed=text)
    print(f"{row['state']:8} filler: {row['filler']!s:35} answer after {row['to_answer_s']}s")

The timing log, as the report will read it (H5 uses the 5 timed runs):

In [ ]:
runs = pd.read_json(log, lines=True)
runs[
    [
        "transcript",
        "state",
        "stt",
        "tts",
        "to_transcript_s",
        "to_filler_s",
        "to_answer_s",
        "graph_s",
        "tts_s",
    ]
]

Talk to it. Set `TALK = True` (needs `SPEECH = True` and microphone permission): press Enter, speak, press Enter.

In [ ]:
TALK = False
if TALK:
    rec = PushToTalk(cfg).record()
    if rec:
        print(vc.turn(rec))

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()